# Graph RMA × Embodiment Generalization — 初步工程计划

## 目标与当前收敛
- **研究目标**：做 `hand × object` 双泛化的 in-hand rotation，但当前核心贡献更偏向 **hand-generalization**，对标 `GET-Zero`。
- **动作空间**：必须保持 **joint space action**，不走 task-space / SE(3) action + IK。
- **任务范围（当前建议）**：先做 `z-axis` rotation；物体侧先从 `cube / cuboid / elongated cuboid / cylinder` 起步，再视结果扩到少量 `YCB family`。
- **工程策略**：先做一个能在 `LeapHand` 跑通的 MVP，但网络结构从一开始就保留扩到 `Allegro / Shadow / LEAP` 的入口。

## 当前最关键的工程判断
1. **不需要改 IsaacLab 底层框架**：`ManagerBasedRLEnv + rl_games dict-mode` 已经足够承载 `policy / critic / priv_info / proprio_hist / embodiment_static` 多路输入。
2. **真正该改的是自定义网络和观测分组**，不是强行把所有东西扁平拼进 `policy` observation。
3. **TRO 的 link encoder 可以借，但不能照搬 patch graph**；**GET-Zero 的 distill 流程可以借，但 robot-only graph 不够**；**HORA 的 stage1/stage2 可以借，但全局 latent 不能吃掉局部因式分解优势**。

## 工程证据汇总（基于现有代码库）

### A. IsaacLab / AnyMani 侧：多路观测输入是现成功能
- `AnyMani/scripts/rl_games/train.py` 已经把 `obs_groups` 与 `concate_obs_group` 透传给 `RlGamesVecEnvWrapper`。
- `IsaacLab/source/isaaclab_rl/isaaclab_rl/rl_games/rl_games.py` 明确支持 `concate_obs_group=False` 的 **Dict mode**，可以把 `obs` 做成 `{policy, priv_info, proprio_hist, embodiment_static, ...}`。
- `AnyMani/source/anymani/mytask/RMA/rma.md` 已有结论：`ManagerBasedRLEnv` 足以做 RMA，不必为了多路输入改成 `DirectRLEnv`。

### B. AnyMani 侧：历史与特权信息已有两条可借路径
- `AnyMani/source/anymani/anymani/tasks/inhand/inhand_env_cfg.py` 已有 `policy/critic` 观测组，以及现成的 tactile sensor 配置。
- `AnyMani/source/anymani/anymani/tasks/direct/leaphand/leaphand_continuous_rot_env.py` 已实现 history buffer（Actor/Critic 分开维护），说明“组件级历史观测”在本仓库里已有人走过。
- 结论：**不缺 history / privileged info / tactile 的环境侧组织能力，缺的是自定义网络如何吃这些路输入。**

### C. HORA 侧：teacher/student 和 adaptation 模块可以直接借训练形状
- `hora/hora/algo/models/models.py`：`env_mlp(priv_info) -> extrin_gt`，`adapt_tconv(proprio_hist) -> extrin`，stage2 只训 adaptation 模块。
- `hora/hora/algo/padapt/padapt.py`：冻结 base actor，只优化 `adapt_tconv`，loss 为 `(e - e_gt)^2`。
- `hora/hora/tasks/allegro_hand_hora.py`：环境里已经维护 `priv_info_buf` 和 `proprio_hist_buf`。
- 结论：**HORA 提供了 object-side latent 的最小训练套路，但它是全局 latent，不够解释 hand-generalization。**

### D. GET-Zero 侧：最值得借的是 custom network + distill 管线
- `get_zero/get_zero/rl/train.py`：通过 `register_network('embodiment_transformer', ...)` 把自定义 network builder 挂进 `rl_games`。
- `get_zero/get_zero/rl/rl_distill.py`：提供 experts → distill → checkpoint evaluation 的现成训练组织方式。
- `get_zero/get_zero/distill/models/embodiment_attention.py`：graph bias（SPD / parent-child / edge encoding）在 transformer attention 里实现，而不是只做 naive concat。
- 结论：**GET-Zero 的工程价值主要在“怎么把自定义 embodiment-aware 网络塞进 rl_games，并做多构型蒸馏”。**

### E. TRO 侧：最值得借的是 link encoder，而不是在线 patch graph
- `TRO-Grasp/model/tro_graph.py`：用 BPS + centroid + scale 做 link encoder，跨 embodiment 复用性很强。
- `TRO-Grasp/model/denoiser.py`：OR/RR graph 很强，但对 per-step policy 太重。
- 结论：**只借 link geometry encoder，别把 patch-level object graph 搬进 20Hz policy。**

## 建议的 MVP 架构（工程可实现版本）

### 核心结构
```mermaid
flowchart TD
    JH[Joint state / action history] --> JD[Dynamic joint tokens]
    UF[URDF / joint limits / rest pose / link geometry] --> ES[Static embodiment tokens]
    JD --> CA0[Cross-Attn 0\nstate queries embodiment]
    ES --> CA0
    CA0 --> CJ[Conditioned joint tokens]

    subgraph Object-side latent inference
        F1[Finger 1 history] --> Z1[local latent z1]
        F2[Finger 2 history] --> Z2[local latent z2]
        F3[Finger 3 history] --> Z3[local latent z3]
        F4[Finger 4 history] --> Z4[local latent z4]
        Z1 --> CA1[Cross-Attn 1 / aggregation]
        Z2 --> CA1
        Z3 --> CA1
        Z4 --> CA1
        CA1 --> ZG[global object memory zg]
    end

    CJ --> HG[Sparse hand graph encoder\n(parent-child / SPD bias)]
    ZG --> CA2[Cross-Attn 2\nobject context -> hand]
    HG --> CA2
    CA2 --> HD[hand-conditioned tokens]
    HD --> AH[Per-joint action head]
```

### 设计原则
1. **joint-space action 不动**：最终输出仍是每关节动作。
2. **hand-generalization 主机制**：`dynamic joint stream × static embodiment stream` 做 cross-attention，不是简单拼接。
3. **object-side 表征**：每根手指先形成 local latent，再聚合出 global memory。
4. **local 主路 / global 残差**：全局 memory 更像 coordination context，不是替代局部因式分解。

### 为什么这个版本适合当前问题
- 比 HORA 多了 embodiment-aware hand side；
- 比 GET-Zero 多了 object-side latent；
- 比 TRO 轻很多，仍有机会跑到控制频率。

### 暂不做的事
- 不引入 task-space action + IK 控制链；
- 不引入在线 object patch graph；
- 不在第一版就强求 full hand×object zero-shot 全开。

## 分阶段实施方案（按工程风险排序）

### Phase 0 — 先把接线打通（不碰 fancy loss）
**目标**：证明 `AnyMani + IsaacLab + rl_games` 能把多路输入喂进自定义网络。
- [ ] 在 `AnyMani` 的 in-hand env 配置里新增观测组：`policy`, `critic`, `proprio_hist`, `priv_info`, `embodiment_static`。
- [ ] 复用 `ManagerBasedRLEnv`，不改 IsaacLab 核心。
- [ ] 参考 `get_zero/get_zero/rl/train.py`，在 `AnyMani/scripts/rl_games/train.py` 里增加 `register_network(...)` 路径。
- [ ] 新建自定义 `NetworkBuilder`（建议放在 `AnyMani/source/anymani/anymani/.../networks/` 下）。
- [ ] 第一个冒烟实验只做：`dynamic joint stream × static embodiment stream` cross-attention + 普通 actor head。

**验收标准**：
- 能正常启动训练；
- `obs` 以 dict 形式进入网络；
- 不比 baseline `actor_critic` 慢到不可接受。

### Phase 1 — 单手型 MVP（LeapHand）
**目标**：先在单手型上验证 object-side 表征是否有用。
- [ ] 任务先固定为 `z-axis rotation`。
- [ ] 物体范围先收在 `cube / cuboid / elongated cuboid / cylinder`。
- [ ] 实现每指 local latent $z_f$ 与全局 memory $z_g$。
- [ ] 先用最简版：`Cross-Attn A (finger -> global) + Cross-Attn B (global -> hand)`。
- [ ] 暂时不强求显式 finger summary token，必要时后补。

**建议监督**：
- local auxiliary：预测本指未来 interaction（contact/slip/force proxy）；
- global auxiliary：预测 privileged object extrinsics/global state；
- 主损失：PPO / imitation loss。

**验收标准**：
- 相比 baseline，在物体变化（长宽比、圆柱/方体）上更稳；
- 训练可收敛；
- inference 结构不爆显存。

### Phase 2 — hand-generalization 主实验
**目标**：把核心贡献抬到 hand-generalization。
- [ ] 引入 `Allegro / Shadow / LEAP` 多手型的静态 embodiment tokens。
- [ ] 先不要求 full zero-shot 全开，先建立 experts / demos。
- [ ] 参考 `get_zero/get_zero/rl/rl_distill.py` 做 experts → universal student 蒸馏。
- [ ] 必要时把 `finger summary layer` 从备选升级成正式模块。

**为什么这阶段才上多手型**：
因为 object-side latent 若一开始就和 hand-generalization 一起全开，排错会非常困难。先用 LeapHand 把 object-side 链条训通，再抬到 multi-hand，会更稳。

### Phase 3 — 对齐最终 paper 目标
**目标**：把 claim 收敛为“hand × object 双泛化，但 hand 更核心”。
- [ ] 先报告 hand-generalization 结果（对标 GET-Zero）；
- [ ] 再报告 object family 扩展结果；
- [ ] 如果 zero-shot 不够强，再讨论 tiny fine-tune / adapter 是否作为补充实验。

## 20Hz+ 推理频率：当前判断
**结论：有希望，但不能现在拍胸脯保证。**

### 为什么我认为有机会
- HORA 的 `adapt_tconv + actor MLP` 本来就是按控制频率跑的。
- GET-Zero 的 transformer policy 是在线策略网络，不是离线生成器。
- TRO 慢的是 patch graph diffusion，不是它的 `link encoder`。
- 我们当前建议的序列长度很小：`16 joints + static embodiment tokens + 4 local latents + 1 global memory`，远小于 TRO 的 object-patch graph。

### 要满足 >20Hz 的工程约束
- **link geometry encoder 预计算**：每个手型只算一次，不在每步重算。
- **静态 embodiment tokens 预计算**：URDF 派生特征在 reset / load hand 时生成。
- **history encoder 小型化**：local latent 用小 TCN/GRU，不做长序列时空 transformer。
- **cross-attention 只用在关键链路**：不要做 full all-to-all multimodal transformer。

## 当前最大的三个工程风险
1. **object-side latent 在单手上学出来后，抬到多手时是否会带手型偏见。**
2. **不同手的 joint semantics 不对齐，simple concat 很可能不够，这正是 cross-attention 设计需要被证明的原因。**
3. **多手型数据与 experts 组织复杂度高，若 distill 管线搭不好，方法会卡在训练工程而不是想法本身。**

## 建议的首个可执行里程碑
在 `LeapHand + cube/cuboid/cylinder` 上，完成一个最小网络：
- dynamic joint tokens
- static embodiment tokens
- per-finger local latent
- one global object memory
- cross-attention 两处
- joint-space action head

如果这个最小系统在单手型上比 baseline 更稳，再抬到 multi-hand；否则不要急着讲双泛化故事。